# AHN window diagnostic — GatedDeltaNet, observe-only (Task #1) — **Google Colab**

Environment port of the Kaggle notebook. **Same validated experiment/diagnostic**
(branch `saadat-pipeline-validation`, HEAD `3800bc9`), embedded as a base64 tarball and
reconstructed into `/content/ahn-mdc`. No GitHub auth, no repo upload.

The Juan-approved matched inference config is baked into the bundled
`src/ahnexp/models.py::_force_window` (**sliding_window=256, sliding_window_type=fixed,
ahn_position=prefix, num_attn_sinks=0**, stale `dy_*` deleted). The diagnostic is
**observe-only** — it never writes `model.config`.

## ⚠️ You need an Ampere+ GPU
flash-attn 2.x kernels are **sm_80+ only** and the AHN model calls flash-attention at
every decode step. Colab **free = T4 (sm_75) — will not work**. Use **Colab Pro / Pro+**
and pick an **L4** or **A100** runtime: *Runtime ▸ Change runtime type ▸ GPU*. Cell 1
hard-stops if the GPU is not sm_80+.

## Run order
| step | cell | note |
|---|---|---|
| A — **GPU gate** | **1** | seconds; hard-stops unless CUDA + compute-capability major ≥ 8 |
| B — reconstruct project | **2** | seconds |
| C — environment | **3** | ~5–10 min; pins torch 2.6 (cu126, CXX11-ABI-TRUE) + **prebuilt** flash-attn wheel, **no compile**; exits non-zero on any import failure |
| **RESTART RUNTIME** | — | *Runtime ▸ Restart session* (torch was replaced on disk) |
| D — verify env | **4** | seconds; imports + a real `flash_attn_func` GPU call |
| E — diagnostic | **5** | first run ~15–20 min (6 GB base download + one-time merge); **exactly 2 generations** |
| F — JSON | **6** | seconds |

tarball sha256: `8d18cc09f28fdc610ec4489fcabfa65967f0eae77e947723a947490aa3b60364`


## A · Cell 1 — inspect the assigned GPU (hard-stop unless sm_80+)


In [ ]:
import subprocess

name, cap, cuda_ok = None, None, False
try:
    import torch
    cuda_ok = torch.cuda.is_available()
    if cuda_ok:
        name = torch.cuda.get_device_name(0)
        cap = torch.cuda.get_device_capability(0)
except Exception as e:
    print("torch probe failed, using nvidia-smi:", e)

if cap is None:
    try:
        out = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=name,compute_cap", "--format=csv,noheader"]
        ).decode().strip().splitlines()[0]
        n, cc = [x.strip() for x in out.split(",")]
        name, cap = name or n, tuple(int(x) for x in cc.split("."))
        cuda_ok = True
    except Exception as e:
        raise SystemExit(f"STOP: no NVIDIA GPU detected ({e}). "
                         "Runtime > Change runtime type > GPU.")

print(f"GPU model          : {name}")
print(f"compute capability : sm_{cap[0]}{cap[1]}")
print(f"CUDA available     : {cuda_ok}")

if not cuda_ok:
    raise SystemExit("STOP: CUDA is not available. Runtime > Change runtime type > GPU.")

if cap[0] < 8:
    raise SystemExit(
        f"\nSTOP: {name} is sm_{cap[0]}{cap[1]}. flash-attn 2.x kernels are sm_80+ only "
        "(Ampere/Ada/Hopper); the AHN model calls flash-attention every decode step. "
        "Colab free = T4 (sm_75). Switch to Colab Pro/Pro+ and select an L4 or A100 "
        "runtime (Runtime > Change runtime type), then re-run this cell."
    )
print("\nGPU OK (sm_80+). Proceed to Cell 2.")


## B · Cell 2 — reconstruct the minimal project


In [ ]:
import base64, hashlib, io, tarfile, pathlib

ROOT = "/content/ahn-mdc"
EXPECT_SHA = "8d18cc09f28fdc610ec4489fcabfa65967f0eae77e947723a947490aa3b60364"

_BLOB = (
    "..."
    "R3NX1ADQAgA="
)

blob = base64.b64decode(_BLOB)
assert hashlib.sha256(blob).hexdigest() == EXPECT_SHA, "tarball checksum mismatch"
pathlib.Path(ROOT).mkdir(parents=True, exist_ok=True)
with tarfile.open(fileobj=io.BytesIO(blob), mode="r:gz") as tf:
    tf.extractall(ROOT)

got = sorted(str(p.relative_to(ROOT)) for p in pathlib.Path(ROOT).rglob("*") if p.is_file())
print(f"reconstructed {len(got)} files under {ROOT}")

for need in ("config/experiment.yaml", "config/facts.yaml", "src/ahnexp/models.py",
             "src/ahnexp/dataset.py", "src/ahnexp/evaluate.py",
             "scripts/diag_ahn_window.py", "scripts/setup_kaggle.sh"):
    assert (pathlib.Path(ROOT) / need).is_file(), f"missing {need}"

m = (pathlib.Path(ROOT) / "src/ahnexp/models.py").read_text()
assert 'model.config.sliding_window_type = matched["sliding_window_type"]' in m
assert "model.config.num_attn_sinks = 0" in m
assert '("dy_sliding_window", "dy_num_attn_sinks")' in m
s = (pathlib.Path(ROOT) / "scripts/setup_kaggle.sh").read_text()
assert 'TORCH_VER="2.6.0"' in s and 'cu126' in s
assert 'FA_VER="2.8.3.post1"' in s
assert 'FLASH_ATTENTION_FORCE_BUILD' not in s
print("OK - matched-config freeze present; setup pins torch 2.6/cu126 + prebuilt flash-attn (no compile).")


## C · Cell 3 — environment (pin torch 2.6/cu126, prebuilt flash-attn, AHN + fla)

Runs the **same** bundled `scripts/setup_kaggle.sh` (environment-neutral; named for its
origin) with `AHNEXP_ROOT`/`AHN_REPO` pointed at `/content`. It pins `torch==2.6.0` from
the **cu126** (manylinux_2_28 / CXX11-ABI-TRUE) index, removes torchvision/torchaudio,
installs the **prebuilt** `flash_attn-2.8.3.post1` torch2.6 abiTRUE wheel by URL
(**no source build**), the Seerkfang `flash-linear-attention` fork, and the ByteDance
AHN package (core only). `transformers` stays pinned at `4.51.0`. If `fla`, `flash_attn`,
or `ahn.transformer.qwen2_ahn` fails to import, the script exits non-zero and prints **no**
success line.


In [ ]:
import os, subprocess
os.environ["AHNEXP_ROOT"] = "/content/ahn-mdc"
os.environ["AHN_REPO"] = "/content/AHN"
rc = subprocess.call(["bash", "/content/ahn-mdc/scripts/setup_kaggle.sh"], env={**os.environ})
print("\nsetup exit code:", rc)
assert rc == 0, ("setup failed - see the FAIL lines above. If torch was already swapped, "
                 "use a fresh runtime: Runtime > Disconnect and delete runtime, then start over.")
print(">>> Runtime > Restart session, then run Cell 4. Do NOT re-run Cells 1-3. <<<")


## ⚠️ RESTART THE RUNTIME NOW

**Runtime ▸ Restart session.** `torch` was just replaced on disk (Colab's build →
2.6.0+cu126); the running runtime still holds the old one in memory.

After restarting, run **Cell 4, 5, 6**. Do **not** re-run Cell 1 / 2 / 3.


## D · Cell 4 — verify the environment (fail fast, real GPU kernel call)


In [ ]:
import importlib, torch, transformers
print("torch        ", torch.__version__, "| cuda", torch.version.cuda,
      "| available", torch.cuda.is_available())
print("gpu          ", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
print("transformers ", transformers.__version__)
assert torch.__version__.startswith("2.6."), (
    f"torch is {torch.__version__} - the runtime still has the old torch; RESTART and re-run Cell 4.")
assert torch.compiled_with_cxx11_abi(), "torch is not CXX11-ABI-TRUE (need the cu126 build)"
assert transformers.__version__ == "4.51.0", transformers.__version__
assert torch.cuda.is_available(), "no CUDA GPU"
cap = torch.cuda.get_device_capability()
print("gpu capability sm_%d%d" % cap)
assert cap[0] >= 8, ("GPU is sm_%d%d - flash-attn 2.x needs sm_80+ (Ampere/Ada/Hopper); "
                     "pick an L4 or A100 Colab runtime." % cap)

for mod in ("triton", "fla", "flash_attn", "ahn.transformer.qwen2_ahn"):
    x = importlib.import_module(mod)
    print(f"import {mod:30} OK  {getattr(x, '__version__', '')}")

# prove the flash-attn CUDA extension actually runs on this GPU (not just imports)
from flash_attn import flash_attn_func
q = torch.randn(1, 8, 2, 16, dtype=torch.float16, device="cuda")
o = flash_attn_func(q, q, q, causal=True)
assert tuple(o.shape) == (1, 8, 2, 16)
print("flash_attn_func on GPU: OK", tuple(o.shape))

# prove the AHN custom Qwen2 classes register
from ahn.transformer.qwen2_ahn import register_customized_qwen2
register_customized_qwen2()
print("register_customized_qwen2: OK")
print("\nENV VERIFIED - safe to run Cell 5.")


## E · Cell 5 — run the observe-only diagnostic

Loads **only GatedDeltaNet**, merges weights once (~6 GB base download, cached under
`/content/ahn-mdc/merged_ckpt`), verifies the custom AHN class + `.ahn` params, builds two
trajectories straddling W=256, **hard-stops if the recurrent one does not cross W**, then
runs **exactly one exact-memory and one recurrent-memory generation**. Observe-only — it
does not modify `model.config`. Same hard gates and output fields as the Kaggle run.


In [ ]:
import subprocess, sys
cmd = [sys.executable, "/content/ahn-mdc/scripts/diag_ahn_window.py",
       "--repo", "/content/ahn-mdc", "--ahn-repo", "/content/AHN"]
print(" ".join(cmd), "\n")
rc = subprocess.call(cmd)
print("\ndiagnostic exit code:", rc, "(0 = PASSED, 2 = INCONCLUSIVE, 1 = hard-stop/error)")


## F · Cell 6 — print the diagnostic JSON


In [ ]:
import pathlib
p = pathlib.Path("/content/ahn-mdc/outputs/diag_ahn_window.json")
print(p.read_text() if p.is_file() else
      "no JSON - the run hard-stopped early; paste the Cell 5 output above.")


## What to paste back for review

Full output of **Cell 1** (GPU), **Cell 4** (env), **Cell 5** (diagnostic), and the
**Cell 6** JSON. Key checks:

* Cell 1: `GPU model` is L4 or A100; `compute capability sm_89` (L4) or `sm_80` (A100)
* Cell 4: `torch 2.6.x`, `flash_attn_func on GPU: OK`, `register_customized_qwen2: OK`
* `[2]` checkpoint pre-override: `sliding_window 256`, `sliding_window_type random`, `ahn_position random`
* `[3]` after `models.load`: `sliding_window 256`, **`sliding_window_type fixed`**, **`ahn_position prefix`**, **`num_attn_sinks 0`**, `dy_* <unset>`, `model.training False`, `effective_window 256`
* `[3b]` `model class ahn.transformer.qwen2_ahn.qwen2_ahn.Qwen2ForCausalLM`; `generation_config.use_cache True`
* `[2] AHN modules` `36 x Qwen2MemDecoderLayer`, `layer[0] .ahn = BaseAHN / fn=GatedDeltaNet`
* `[6]` the two `model_tokens_after_target` (≈49 and ≈2100)
* `HARD GATE` line = `PASS`
* `[9]` both trial rows: `prediction`, `answer_canonical`, `correct`, `malformed`, `abstained`, `n_new_tokens`, **`ahn_layer0_num_cached_tokens`** (0 exact / ~1850+ recurrent), **`ahn_kernel_forward_calls`** (0 exact / ≥1 recurrent)
* `[10]` verdict block + `DIAGNOSTIC PASSED` / `INCONCLUSIVE`

If Cell 5 exits early, paste what printed — the hard gate stopping is a valid result. **Do not run the experimental grid regardless of outcome.**
